## Create Input Interventions

**General note:** A more detailed analysis and interpretation of the results obtained from the controlled input interventions, as well as the rationale for the chosen procedures, is provided in the final paper. This notebook focuses primarily on the implementation, validation, and documentation of the intervention procedure.

This notebook creates controlled variants of the held-out human-annotated test set for the shortcut analysis.

Each intervention modifies one specific source of model input while keeping the remaining information unchanged. This allows later model evaluation to measure whether predictions depend on the supplied target, explicit candidate mentions, or other lexical cues.

The intervention datasets themselves are created from the held-out human-annotated test set. Whenever an intervention requires data-driven rule construction, such as identifying candidate-name variants or lexical cues, these rules are derived exclusively from the training data and frozen before they are applied to the test set.

This separation prevents the held-out test data from influencing the construction of the intervention rules.


The original human-annotated test set is loaded from the output of our preprocessing.

All intervention datasets are stored separately under `data/interventions/`

---

Start by loading the human annotated subset and inspecting the dataset structure:

In [504]:
import pandas as pd

human_test = pd.read_parquet('../data/preprocessed/human_test.parquet')


print(f"Number of rows: {len(human_test):,}")
print(f"Columns: {human_test.columns.tolist()}")

human_test.head(3)

Number of rows: 890
Columns: ['UserId', 'TargetEntity', 'StanceLabel', 'ContextPosts']


,UserId,TargetEntity,StanceLabel,ContextPosts
0,186791,Trump,Against,"[{'Content': 'Lmao #fucktrump', 'PostTime': '2..."
1,88089,Trump,Against,"[{'Content': '""I can't get laid because of tax..."
2,254114,Trump,Against,"[{'Content': 'This has always been a problem, ..."


Check which target and stance values occur in the held out test set:

In [505]:
print("Target values:")
display(human_test["TargetEntity"].value_counts(dropna=False))

print("Stance labels:")
display(human_test["StanceLabel"].value_counts(dropna=False))

Target values:


TargetEntity
Trump     445
Harris    445
Name: count, dtype: int64

Stance labels:


StanceLabel
Against    440
Favor      235
Neither    215
Name: count, dtype: int64

Look at a context posts example:

In [506]:
print("First ContextPosts entry:")
print(human_test["ContextPosts"][0])

First ContextPosts entry:
[{'Content': 'Lmao #fucktrump', 'PostTime': '2024-11-25T02:39:25.551Z', 'IsAmbiguous': False}
 {'Content': 'I don’t know about anyone else but I get a kick out of people of color voting for Trump. Do you people  realize he only sees one color WHITE. He only sees one brand the RICH brand.', 'PostTime': '2024-11-24T14:56:33.890Z', 'IsAmbiguous': False}
 {'Content': 'If you are a republican and voted for Kamala I want to follow you. If you’re a republican and voted for Trump I want to say it without saying it.', 'PostTime': '2024-11-24T14:32:20.729Z', 'IsAmbiguous': False}
 {'Content': 'More people voted against Trump than voted for him.', 'PostTime': '2024-11-24T03:48:21.454Z', 'IsAmbiguous': False}
 {'Content': 'I don’t know about anyone else but I am never eating at Mc Donald’s again. They let Trump touch our fries.', 'PostTime': '2024-11-23T20:05:10.285Z', 'IsAmbiguous': False}
 {'Content': 'For as long as I live, I will never understand how anyone could have

---

## **Intervention 1: Target Masking**

The first intervention removes the identity of the supplied target while preserving the input structure.

For every test example:

- `TargetEntity` is replaced with the constant placeholder `[TARGET_REMOVED]`.
- `ContextPosts` remain completely unchanged.

This intervention tests whether model behavior changes when the identity of the supplied target is unavailable.

In [507]:
target_masked_test = human_test.copy()

target_masked_test["TargetEntity"] = "[TARGET_REMOVED]"

Sanity check whether intervention was successful:

In [508]:
target_masked_test.head()

,UserId,TargetEntity,StanceLabel,ContextPosts
0,186791,[TARGET_REMOVED],Against,"[{'Content': 'Lmao #fucktrump', 'PostTime': '2..."
1,88089,[TARGET_REMOVED],Against,"[{'Content': '""I can't get laid because of tax..."
2,254114,[TARGET_REMOVED],Against,"[{'Content': 'This has always been a problem, ..."
3,77504,[TARGET_REMOVED],Against,[{'Content': 'Tomorrow morning Democracy Docke...
4,132412,[TARGET_REMOVED],Against,"[{'Content': 'We need protection from Trump.',..."


In [509]:
# All targets were replaced correctly
assert target_masked_test["TargetEntity"].eq("[TARGET_REMOVED]").all()

# Everything except TargetEntity remains unchanged
unchanged_columns = [column for column in human_test.columns if column != "TargetEntity"]

for column in unchanged_columns:
    assert target_masked_test[column].equals(human_test[column])

Finally save the target-masked test set:

In [399]:
from pathlib import Path

Path("../data/interventions").mkdir()

target_masked_test.to_parquet("../data/interventions/human_test_target_masked.parquet", index=False)

---

## **Intervention 2: Target Swapping**

The second intervention changes only the supplied target while keeping the remaining input unchanged.

For every test example:

- `Trump` is replaced with `Harris`.
- `Harris` is replaced with `Trump`.
- `ContextPosts` remain completely unchanged.

Unlike target masking, this intervention provides the model with a valid but opposite target identity.

This intervention tests whether model predictions are sensitive to changes in the supplied target.

Important: The original StanceLabel is retained in the target-swapped dataset only to preserve the pairing with the corresponding original example. After changing the target, this label must not be interpreted as a gold label for the modified input. Target swapping is therefore evaluated through changes in model predictions and predicted class probabilities rather than classification performance against the retained label.

In [510]:
target_swapped_test = human_test.copy()

target_swap = {
    "Trump": "Harris",
    "Harris": "Trump"
}

target_swapped_test["TargetEntity"] = (target_swapped_test["TargetEntity"].map(target_swap))

Sanity check whether the intervention was successful:

In [511]:
target_swapped_test

,UserId,TargetEntity,StanceLabel,ContextPosts
0,186791,Harris,Against,"[{'Content': 'Lmao #fucktrump', 'PostTime': '2..."
1,88089,Harris,Against,"[{'Content': '""I can't get laid because of tax..."
2,254114,Harris,Against,"[{'Content': 'This has always been a problem, ..."
3,77504,Harris,Against,[{'Content': 'Tomorrow morning Democracy Docke...
4,132412,Harris,Against,"[{'Content': 'We need protection from Trump.',..."
5,7049,Harris,Against,"[{'Content': '""Muslims for Trump"" is on par wi..."
6,111658,Harris,Against,[{'Content': 'Donald Trump is suffering from d...
7,12476,Harris,Against,[{'Content': 'Donald Trump is the stupidest mo...
8,182701,Harris,Against,[{'Content': 'I think a lot of us feel that Tr...
9,239303,Harris,Against,[{'Content': 'More people voted against Trump ...


In [512]:
# All targets were swapped correctly
expected_swapped_targets = human_test["TargetEntity"].map(target_swap)

assert target_swapped_test["TargetEntity"].equals(expected_swapped_targets)

# Everything except TargetEntity remains unchanged
unchanged_columns = [column for column in human_test.columns if column != "TargetEntity"]

for column in unchanged_columns:
    assert target_swapped_test[column].equals(human_test[column])

Finally save the target-swapped test set:

In [513]:
target_swapped_test.to_parquet("../data/interventions/human_test_target_swapped.parquet", index=False)

---

## **Intervention 3: Candidate-Mention Masking**

The third intervention removes explicit name-based references to Donald Trump and Kamala Harris from the retrieved context posts while leaving the supplied target and all remaining input information unchanged.

All candidate references are replaced with the same neutral placeholder:

- `Trump` → `[CANDIDATE]`
- `Donald Trump` → `[CANDIDATE]`
- `Kamala` → `[CANDIDATE]`
- `Kamala Harris` → `[CANDIDATE]`

The same placeholder is deliberately used for both candidates. Candidate-specific placeholders such as `[TRUMP]` and `[HARRIS]` would preserve exactly the identity information that the intervention is intended to remove.

References to both candidates are masked regardless of the supplied `TargetEntity`. This prevents the non-target candidate from remaining as an alternative identity cue.

When a candidate name occurs inside a larger expression, only the candidate-identifying component is replaced whenever possible. For example:

- `anti-Trump` → `anti-[CANDIDATE]`
- `#fucktrump` → `#fuck[CANDIDATE]`
- `Biden-Harris` → `Biden-[CANDIDATE]`
- `#HarrisWalz` → `#[CANDIDATE]Walz`

This preserves surrounding lexical and stance-related information while removing the explicit candidate identity.

The masking rule is constructed exclusively from the training corpus and is frozen before being applied unchanged to the held-out human-annotated test set.

The intervention therefore tests whether the models can still infer stance from indirect contextual evidence when explicit candidate-name information is unavailable in the retrieved posts.

### **Operational definition of a candidate mention**

For this intervention, a candidate mention is defined as an explicit name-based surface reference to Donald Trump or Kamala Harris.

The masking procedure therefore targets:

- complete candidate names,
- candidate first and last names,
- name-based hashtags,
- name-based handles or usernames (A handle here means a social media username/account name, typically introduced with @),
- compounds and affixed forms containing candidate-name components.

The intervention does **not** attempt to remove every expression from which candidate identity could potentially be inferred. Indirect political references such as `MAGA`, `Republican`, `Democrat`, `Vance`, `Walz`, policy terms, ideological expressions, or generic titles remain unchanged unless they explicitly contain one of the candidate-name components.

This restriction is intentional. Candidate-Mention Masking measures reliance on **explicit candidate identity**, while broader political and label-correlated expressions are examined separately in the lexical-cue intervention.

### **Training-only construction of the masking rule**

The final masking rule is derived exclusively from the training corpus rather than from the held-out test set.

The procedure is intentionally recall-oriented (find as many potentially relevant candidate mentions as possible, even if that initially includes some false positives):

1. candidate-name components are used as broad discovery anchors;
2. all training surface forms containing these anchors are collected;
3. the discovered forms are manually reviewed for systematic false positives and ambiguous uses;
4. an additional context audit identifies cases in which an otherwise valid candidate surname belongs to another person;
5. the resulting exception and protection rules are frozen;
6. the frozen rule is subsequently applied unchanged to the held-out test set.

The review records **exceptions** to the broad matching rule rather than manually constructing a complete whitelist of candidate expressions. This allows the final matcher to generalize to previously unseen compounds or hashtags while explicitly protecting forms that were shown to be unreliable during the training-data review.

### **Load the training data**

Candidate-matching rules are derived exclusively from the preprocessed training set.

In [514]:
train = pd.read_parquet("../data/preprocessed/train.parquet")

print(f"Number of training examples: {len(train):,}")
print(f"Columns: {train.columns.tolist()}")

Number of training examples: 12,834
Columns: ['UserId', 'TargetEntity', 'StanceLabel', 'ContextPosts']


### **Extract the training post texts**

The text contents of all retrieved training posts are extracted for candidate-variant discovery.

In [515]:
# Extract the text of every retrieved training post
training_post_texts = [
    post["Content"]
    for context_posts in train["ContextPosts"]
    for post in context_posts
    if post.get("Content")
]

print(f"Number of retrieved training posts: {len(training_post_texts):,}")

Number of retrieved training posts: 95,896


### **Training-only candidate-form discovery**

Four explicit candidate-name components are used as discovery anchors:

- `trump`
- `donald`
- `harris`
- `kamala`

These anchors are not themselves a final dictionary of candidate mentions. They are used to discover candidate-like surface forms occurring in the training corpus.

The search is case-insensitive and captures complete surface tokens containing at least one anchor. This includes forms such as hashtags, compounds, affixed expressions and username-like strings.

For example:

- `Trump`
- `anti-Trump`
- `Trump2024`
- `#NeverTrump`
- `KamalaHQ`
- `Biden-Harris`

The procedure does not claim to discover arbitrary misspellings that contain none of the four anchors. It specifically discovers surface forms containing at least one exact candidate-name component.

In [516]:
search_anchors = {
    "Trump": ["trump", "donald"],
    "Harris": ["harris", "kamala"],
}

The discovery anchors are combined into a single case-insensitive regular expression.

The expression captures the complete surrounding surface token rather than only the anchor itself. This is necessary because the masking procedure should preserve the non-name component of expressions such as `anti-Trump` or `#fucktrump`.

The discovery expression is deliberately broad. Consequently, it also retrieves non-candidate strings such as `McDonalds`, `Harrison`, or `trumpet`. These are not treated as errors in discovery; they are expected consequences of the high-recall search and are handled explicitly during the subsequent review.

In [517]:
import re

# Collect all candidate-name discovery anchors
all_anchors = [anchor for anchors in search_anchors.values() for anchor in anchors]

# Escape anchors before inserting them into the regular expression
anchor_pattern = "|".join(re.escape(anchor) for anchor in all_anchors)

# Match complete surface tokens that contain at least one candidate-name anchor
# This includes ordinary tokens, hashtags, handles, compounds, and suffix variants
variant_pattern = re.compile(rf"(?<!\w)[#@]?[A-Za-z0-9_-]*(?:{anchor_pattern})[A-Za-z0-9_-]*(?!\w)", flags=re.IGNORECASE,)

### **Collect discovered surface forms and training contexts**

Every retrieved training post is scanned with the candidate discovery pattern.

For each unique surface form, two pieces of diagnostic information are retained:

- the number of retrieved training posts in which the form occurs;
- up to three example posts containing the form.

A surface form is counted at most once per post, even if it occurs multiple times within that post.

The examples are retained solely to support the manual training-data review of false positives and ambiguous forms.

In [518]:
variant_info = {}

for text in training_post_texts:

    # Find all matching variants in the current post.
    # Using a set ensures that the same variant is counted only once per post.
    variants_in_post = {
        match.group(0).lower()
        for match in variant_pattern.finditer(text)
    }

    for variant in variants_in_post:

        # Initialize metadata for variants seen for the first time
        if variant not in variant_info:
            variant_info[variant] = {
                "post_count": 0,
                "examples": []
            }

        # Count in how many posts the variant occurs
        variant_info[variant]["post_count"] += 1

        # Store up to three example posts for later manual inspection
        if len(variant_info[variant]["examples"]) < 3:
            variant_info[variant]["examples"].append(text)


print(f"Potential surface forms discovered: {len(variant_info):,}")

Potential surface forms discovered: 782


### **Construct the candidate-form audit table**

The discovered expressions are collected in a structured audit table.

For each surface form, the table records:

- the candidate suggested by the discovery anchors;
- the observed surface form;
- the surface-form category (`variant`, `hashtag`, or `handle`);
- the number of retrieved training posts containing the form;
- up to three training examples.

If a surface form contains anchors associated with both candidates, it is labeled `BOTH`. Such forms are not considered ambiguous merely because they mention both candidates: both candidate components can be masked independently.

The audit table provides the evidence used for the subsequent manual exception review and is saved before any manual decisions are applied.

For `infer_candidate`: Each discovered surface form is assigned to the candidate whose discovery anchors occur in the expression.

If anchors associated with exactly one candidate occur, that candidate is assigned. If anchors associated with both candidates occur, the expression is labeled `BOTH`.

Because all expressions were discovered using at least one candidate-name anchor, unmatched expressions are not expected. `UNRESOLVED` is retained only as a diagnostic safeguard.

In [519]:
def infer_candidate(phrase):
    # Normalize the phrase so anchor matching is case-insensitive
    phrase = phrase.lower()

    # Identify which candidates have at least one anchor in the phrase
    matched_candidates = [
        candidate
        for candidate, anchors in search_anchors.items()
        if any(anchor in phrase for anchor in anchors)
    ]

    # Exactly one candidate is referenced
    if len(matched_candidates) == 1:
        return matched_candidates[0]

    # Name components of both candidates occur in the same expression
    if len(matched_candidates) > 1:
        return "BOTH"

    # This should not occur because candidate forms were discovered
    # using the same search anchors
    return "UNRESOLVED"

For: `infer_variant_type`: Each detected phrase is classified based on its prefix.  
Phrases starting with `#` are labeled as `hashtag`, phrases starting with `@` as `handle`, and all remaining phrases as `variant`.

In [520]:
def infer_variant_type(phrase):
    # Classify phrases starting with "#" as hashtags
    if phrase.startswith("#"):
        return "hashtag"

    # Classify phrases starting with "@" as handles/usernames
    elif phrase.startswith("@"):
        return "handle"

    # All other detected phrases are treated as general variants
    else:
        return "variant"

Create the table: The discovered expressions and their metadata are combined into the candidate-form audit table.

For each expression, the inferred candidate, surface-form type, frequency across retrieved training posts, and up to three example contexts are retained. The table is sorted by frequency to support the subsequent manual review.

In [521]:
# Collect detected candidate variants and their metadata
variant_rows = []

for phrase, info in variant_info.items():
    # Retrieve example posts in which the variant occurred
    examples = info["examples"]

    # Create one row per detected variant
    variant_rows.append({
        "candidate": infer_candidate(phrase),      # Trump, Harris, BOTH, or UNRESOLVED
        "phrase": phrase,                          # Detected phrase
        "type": infer_variant_type(phrase),        # hashtag, handle, or variant
        "post_count": info["post_count"],          # Number of posts containing the phrase
        "example_1": examples[0] if len(examples) > 0 else None,
        "example_2": examples[1] if len(examples) > 1 else None,
        "example_3": examples[2] if len(examples) > 2 else None,
    })

# Convert collected variants into a DataFrame,
# sort by frequency, and reset the index
candidate_variants = (
    pd.DataFrame(variant_rows)
    .sort_values("post_count", ascending=False)
    .reset_index(drop=True)
)

candidate_variants.head(10)

,candidate,phrase,type,post_count,example_1,example_2,example_3
0,Trump,trump,variant,35454,"I don't see how you read ""We are withholding o...","I don't see how you read ""We are withholding o...",Just in case you thought you were doing someth...
1,Harris,harris,variant,9871,"I don't see how you read ""We are withholding o...","I mean so do I, there's several people I'd wan...",All my doom and gloom aside. If Harris needs a...
2,Harris,kamala,variant,8371,"I don't see how you read ""We are withholding o...","I don't see how you read ""We are withholding o...",Kamala Harris is very good at this.
3,Trump,donald,variant,7605,"I don't see how you read ""We are withholding o...","I don't see how you read ""We are withholding o...","Not to get too political on here, but I really..."
4,Trump,#trump,hashtag,936,Trump’s Effort to Kill Off #MeToo—And the Wome...,Election 2024: Presidential results\nFormer Pr...,Dus June Kii Raat 2024 Hindi Season 2\n#Casa #...
5,Trump,trumps,variant,522,donald trumps hiring people with the one quali...,You all do realize that this point we could li...,Seems like sexual assault is a pre-requisite t...
6,Harris,#kamalaharris,hashtag,297,"“Kamala, you didn’t just run—you thundered. Yo...",I miss #KamalaHarris and #TimWalz \n\nTheir jo...,Heroine of the day: #KamalaHarris
7,Trump,mcdonald,variant,223,Trump propagandists went full North Korea over...,"If You Want McDonald to go Prison, Vote for Ha...",Ask yourself why Trump was like a dog with a b...
8,Trump,#fucktrump,hashtag,208,#FuckTrump #MAGA #MAGAts #MAGACultMorons,#FuckTrump #MAGA #MAGAts #MAGACultMorons,I only wish that tRumpublicans and the Conserv...
9,Trump,#donaldtrump,hashtag,187,Who will become the next US President? The one...,THAT'S RIGHT!!! WE WANT 'NOTHING' TO DO WITH T...,KAMALA HARRIS 2024!!!! \n🇺🇲 ❤️\n\nKamala Harri...


### **Save the candidate-form audit**

The complete set of discovered candidate-related surface forms is retained as an audit file.

The audit records the frequency and training examples underlying the matching-rule review. This makes the derivation of the final rule reproducible without hard-coding every observed candidate-related expression into the intervention itself.

In [522]:
# Save the complete training-derived candidate-form audit
candidate_variants.to_csv(
    "../data/interventions/candidate_variant_audit.csv",
    index=False,
)

### **Separate surface-form categories for review**

The discovered forms are divided into ordinary variants, hashtags, and handles because these categories exhibit different types of false positives.

Ordinary forms may contain unrelated lexical items such as `trumpet` or names such as `Harrison`. Hashtags can combine several words into a single token and therefore require separate inspection.

The same discovery rule is used for all categories; the separation is only used to make the manual audit clearer and more reproducible.

In [523]:
# Split the discovered forms into ordinary variants, hashtags, and handles
normal_variants = (candidate_variants[candidate_variants["type"] == "variant"].copy().reset_index(drop=True))

hashtags = (candidate_variants[candidate_variants["type"] == "hashtag"].copy().reset_index(drop=True))

handles = (candidate_variants[candidate_variants["type"] == "handle"].copy().reset_index(drop=True))


# Report how many discovered forms belong to each category
print(f"Normal variants: {len(normal_variants):,}")
print(f"Hashtags:        {len(hashtags):,}")
print(f"Handles:         {len(handles):,}")

Normal variants: 296
Hashtags:        486
Handles:         0


No candidate-containing handles were discovered in the retrieved training posts. Consequently, no handle-specific exception list is required for this dataset.

### **Manual review of ordinary candidate-like variants**

The 296 ordinary surface forms discovered from the training corpus are inspected together with their frequencies and example contexts.

The review records two kinds of exceptions.

**False positives** are forms for which the candidate-name anchor is part of another word, name, or place and therefore does not constitute a reference to Donald Trump or Kamala Harris. Examples include `mcdonalds`, `harrison`, and `harrisburg`.

**Ambiguous forms** are surface forms that can refer to a candidate in some political contexts but also have a plausible non-candidate lexical meaning. Because the final masking rule is applied automatically and without test-set-specific contextual judgement, these forms are conservatively left unchanged.

All other discovered ordinary variants are accepted by default as explicit candidate-name references.

The lists below therefore do not represent an externally compiled dictionary. They are the documented exception decisions resulting from manual inspection of the training-derived candidate-form audit.

In [524]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(normal_variants[["candidate", "phrase", "post_count", "example_1", "example_2", "example_3",]])

,candidate,phrase,post_count,example_1,example_2,example_3
0,Trump,trump,35454,"I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement.","I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement.",Just in case you thought you were doing something other than helping elect Trump by voting for Jill Stein
1,Harris,harris,9871,"I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement.","I mean so do I, there's several people I'd want over Harris. That's...not the same as refusing to vote for or support Harris in THIS election.\n\nSomeone criticizing Harris for specific positions does not automatically equal ""will not vote for her""","All my doom and gloom aside. If Harris needs a boring, bland white guy to balance the ticket...I'm available."
2,Harris,kamala,8371,"I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement.","I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement.",Kamala Harris is very good at this.
3,Trump,donald,7605,"I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement.","I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement.","Not to get too political on here, but I really hope that Donald Trump does not win the presidential election. \n\nI think that'd be bad!"
4,Trump,trumps,522,donald trumps hiring people with the one qualification needed: need to be pedophile,You all do realize that this point we could literally get a restraining order against many of trumps picks to not be allowed near schools. You do realize that. \n\nLETS MAKE AMERICA GREAT. \n\nYeah NOT THIS WAY,Seems like sexual assault is a pre-requisite to be on Trumps cabinet
5,Trump,mcdonald,223,"Trump propagandists went full North Korea over his McDonald's stunt, but the story right now is all about his refusal to answer questions about the minimum wage.\n\nOn the pod, @anonymous is great on how Harris's life experiences inform her policy agenda:\nhttps://anonymous","If You Want McDonald to go Prison, Vote for Harris and Democrats.",Ask yourself why Trump was like a dog with a bone about Kamala Harris’ McDonald’s job. He couldn’t go a day without bitching about it and yet not a word about an assassination attempt on his life that left two dead.
6,Trump,trumpism,154,"Jasmine Crockett is challenging Debbie Dingell for chairmanship of the Democratic Policy and Communications Committee. \n\nI’m sorry but this is a no-brainer. If we are going to defeat Trumpism we need to turn the page on our defunct comms strategies.\n\nLet’s go, Jasmine!",I want my first viral post on Bluesky to be about how much I hate Trumpism.\n\nIt’s ruined this country.,I can’t help that Trumpism is a cult I can tell sane people to not focus on fringe issues.
7,Trump,trumpers,141,Found out Nara Smith & her toothpick eating husband are trumpers 👎🏻,Trumpers are currently facing a very rude awakening…,"Just as there are ""Never Trumpers"" the there are also ""Never Trudeau""\n\nJustin Trudeau could implement

In [525]:
# Expressions that contain a discovery anchor but clearly refer to another person, place, organization, or lexical item rather than Donald Trump or Kamala Harris
normal_false_positives = {
    # McDonald / McDonald's rather than Donald Trump
    "mcdonalds",
    "macdonalds",
    "mcdonalds-employee",
    "mcdonalds-",

    # Other people whose names happen to contain "donald"
    "donalds",           # Byron Donalds
    "donaldson",         # Donaldson
    "erictrump952789",   # Eric Trump rather than Donald Trump
    "maryltrump",        # Mary L. Trump rather than Donald Trump

    # Other people whose names happen to contain "harris"
    "harrison",          # Harrison Ford
    "dncjamieharrison",  # Jamie Harrison

    # Place names containing "harris"
    "harrisburg",
    "harrisonvile",
}


# Expressions that are candidate-related in at least some observed training contexts but whose surface form also has a plausible non-candidate meaning. These should not be automatically masked without additional contextual disambiguation.
normal_ambiguous = {
    
    # one observed context that may use it as candidate-directed wordplay
    "mcdonald",

    # Ordinary English verb forms that can also be used as Trump puns
    "trumped",
    "trumping",

    # Ordinary lexical items that are used as Trump-related
    # nicknames/puns in the observed political contexts
    "trumpet",
    "trumpeter",
    "trumpeteer",
    "strumpet",
    "fucktrumpet",
    "trumpery",

    # Unclear username-like construction; the surface form alone
    # does not establish a reference to Donald Trump
    "donaldsoros",
}

### **Manual review of candidate-like hashtags**

Hashtags are reviewed independently because several words can be concatenated into a single surface token.

The same decision rule is used as for ordinary variants:

- hashtags that clearly refer to something other than the two candidates are recorded as false positives;
- hashtags whose interpretation cannot be determined reliably are recorded as ambiguous;
- all remaining hashtags are accepted as candidate-name references.

Importantly, stance-bearing information surrounding the candidate name is not treated as a reason for exclusion. For example, `#fucktrump` and `#NeverTrump` remain candidate references because the candidate-identifying component can be removed while the surrounding stance information is preserved.

The decisions below were derived exclusively from the displayed training-data hashtag audit.

In [526]:
# Inspect candidate-related hashtags from most to least frequent
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(hashtags[["candidate", "phrase", "post_count", "example_1", "example_2", "example_3",]])

,candidate,phrase,post_count,example_1,example_2,example_3
0,Trump,#trump,936,Trump’s Effort to Kill Off #MeToo—And the Women Who Help Him\n#Trump,"Election 2024: Presidential results\nFormer President Donald Trump will defeat Vice President Kamala Harris in a historic political comeback, recapturing the White House following an election loss in 2020, CNN projects.\n#Trump#EleicoesEUA#Trump#SuperStarHongjoongDay\nhttps://anonymous",Dus June Kii Raat 2024 Hindi Season 2\n#Casa #Jean #trump#Estados #Biden#RoçaAFazenda#Cuba#Putin#MAGA#Musk\nstreamnscreen.blogspot.com
1,Harris,#kamalaharris,297,"“Kamala, you didn’t just run—you thundered. You made racism flinch, sexism stammer, and the sky crack open for the next. Even without the win, you rose, showed us how to carve a path with nothing but grit and grace. The climb is ours now, and we will not stop.”©️MustafaSantiagoAli #Poem #KamalaHarris","I miss #KamalaHarris and #TimWalz \n\nTheir jokes, the joy, the ""mind your own damm business"" comments. \n\nJust realized I miss it. \n\n#Resistance \n#FBR",Heroine of the day: #KamalaHarris
2,Trump,#fucktrump,208,#FuckTrump #MAGA #MAGAts #MAGACultMorons,#FuckTrump #MAGA #MAGAts #MAGACultMorons,I only wish that tRumpublicans and the Conservative Christians would believe in this philosophy. \n#transgenderrights #fucktrump #fuckproject2025
3,Trump,#donaldtrump,187,Who will become the next US President? The one who “worked” there when it was actually closed or the one who worked there to pay for her student loan?\n\nhttps://anonymous\n\n#Obama #KamalaHarris #DonaldTrump #McDonalds,THAT'S RIGHT!!! WE WANT 'NOTHING' TO DO WITH THEM! THEY'RE THE ENEMY AS FAR AS I'M CONCERNED!!\n\nCharles Barkley agrees it’s ‘crazy’ Kamala voters want nothing to do with Trump voters #DonaldTrump #KamalaHarris #Bluestates #Basketball #Presidentialelection\r\nhttps://anonymous,KAMALA HARRIS 2024!!!! \n🇺🇲 ❤️\n\nKamala Harris counters Trump's message that the president should have a say in Fed decisions #DonaldTrump #KamalaHarris #Interestrates #Presidentialelection\r\nhttps://anonymous
4,Harris,#harriswalz2024,165,Not much of a meme poster but this one hits the mark. \n\n#HarrisWalz2024,https://anonymous\n\n#HarrisWalz2024 \n#DemocracyMatters,#WeBackKamalaHarris🇺🇲🌊💙🇵🇷 #HarrisWalz2024 🇺🇲🌊💙🇵🇷🇺🇦🇯🇴🤲🏿🌿 #EbonyAmericaVoteBlue2024🌊💙🌊💙 #WomansRightToChoose💪🏾💅🏾#TaylorSwiftForKamalaWalz2024 🐈💅🏾💪🏾
5,Harris,#kamala,132,We needed to have #msm #celebration of #JoeBiden #Kamala instead of climbing up Trumps butt parroting him constantly. You will never see them do this with #DoofusDrumpf.,#Trump2024 #project25 #DOGE #Kamalawalz #Kamala #TamponTimmy #swifties4kamala #Gaysforpalestine #AOC #JDVance #JDVance2028 #DNC #Trans,#Trump2024 #project25 #DOGE #Kamalawalz #Kamala #TamponTimmy #swifties4kamala #Gaysforpalestine #AOC #JDVance #JDVance2028 #DNC #Trans
6,Harris,#harris,86,"Wait, was Kamala going to have sex offenders and traffickers in her cabinet? Was she going to tariff goods to the point that companies would slash thousands of jobs and raise prices before she took office? \n\nShe's supposed to be ""just as bad,"" right?\n\n#Kamala #Harris #USA #Trump #RIPtoMyWallet #sad","CNN’s Kaitlin Collins: “With this speech and her earlier call conceding the race, #Harris is affording #Trump what he did not afford to the incoming Biden-Harris administration: a commitment to a peaceful transition of power and acknowledgment of a legitimate victory.”",""" #Kamala #Harris Goes BOSS: Calls #Trump THE F WORD!"" \n\nhttps://anonymous #harriswalz #kamalaharris #harriswalz2024 #voteblue #vote"
7,Trump,#nevertrump,70,Happy Friday friends 👋\n\nIf you want to meet more more #NeverTrump friends drop a 💙\nI random boost my followers when I see they participate following people to make this process work.\n\nI will post a schedule later for our combined parties tonight so check back! 🎉\n\n#Trumpless #signalBoost\nRepost,Happy Thursday friends 👋\nIf you want to meet more #NeverTrump friends drop a 💙\n\nI 

In [527]:
hashtag_false_positives = {
    # McDonald / McDonald's
    "#mcdonalds",
    "#mcdonald",

    # Other people
    "#byrondonalds",
    "#donaldtrumpjr",
    "#melaniatrump",
    "#harrisonford",
    "#harrison",

    # Non-candidate lexical uses
    "#trumpet",
    "#lovetrumpshate",
}

# No hashtag remained genuinely ambiguous after inspection
# of the observed training contexts.
hashtag_ambiguous = set()

### **Focused audit for non-candidate person names**

The surface-form audit alone cannot resolve every false candidate match.

For example, the token `Trump` is normally a valid reference to Donald Trump and must therefore remain an accepted surface form. However, in an expression such as `Mary Trump`, the same token is the surname of another person and should not be removed by this intervention.

The same problem occurs for other people whose surname is `Harris`.

A second, focused training-only audit therefore examines the immediate lexical context surrounding occurrences of `Trump` and `Harris`. Its purpose is specifically to identify multi-token names in which an otherwise valid candidate surname belongs to another person.

This is distinct from the surface-form false-positive lists:

- `Harrison` or `Donalds` can be excluded based on the token itself;
- `Mary Trump` or `Sam Harris` require protection of the complete multi-token expression because `Trump` and `Harris` remain valid candidate references in other contexts.

Common candidate-related constructions such as `Trump administration`, `Trump voters`, `Harris campaign`, or `Trump Tower` are **not** protected because the surname still functions as an explicit candidate-identity cue in those expressions.

Therefore the following code extracts all two-word contexts containing Trump or Harris, normalizes them, and counts how frequently each unique expression occurs in the training corpus.

In [528]:
# inspect immediate neighbours of Trump/Harris in the TRAINING data only: X Trump / X Harris and Trump X / Harris X

adjacent_candidate_context_pattern = re.compile(
    r"\b(?:[\w.'’-]+\s+(?:Trump|Harris)|(?:Trump|Harris)\s+[\w.'’-]+)\b",
    flags=re.IGNORECASE
)

adjacent_contexts = []

for text in training_post_texts:
    if not isinstance(text, str):
        continue

    for match in adjacent_candidate_context_pattern.finditer(text):
        adjacent_contexts.append(match.group(0))


adjacent_context_counts = (
    pd.Series(adjacent_contexts, dtype="string")
    .str.strip()
    .str.lower()
    .value_counts()
    .rename_axis("expression")
    .reset_index(name="count")
)

adjacent_context_counts.to_csv(
    "../data/interventions/candidate_adjacent_context_audit.csv",
    index=False,
)

display(adjacent_context_counts.head(100))

,expression,count
0,donald trump,7169
1,kamala harris,6190
2,for trump,1999
3,that trump,1329
4,the trump,1130
5,of trump,1085
6,trump is,1056
7,to trump,789
8,a trump,620
9,with trump,576


### **Protect references to other people**

Manual inspection of the focused training-context audit identified a small set of multi-token names in which `Trump` or `Harris` refers to someone other than the two target candidates.

These complete expressions are protected temporarily before candidate masking and restored afterwards.

This protection is deliberately narrow. It is used only where the candidate-name component belongs to another person's name. Candidate-related constructions such as `Trump administration` or `Harris campaign` remain maskable.

The protection set is derived exclusively from the training-data audit and is frozen before application to the test set.

In [529]:
# Multi-token names containing an exact candidate-name component but referring to people other than Donald Trump or Kamala Harris.

protected_non_candidate_mentions = {
    # Other Trump family members / people named Trump
    "donald trump jr.",
    "donald trump jr",
    "trump jr.",
    "trump jr",
    "melania trump",
    "eric trump",
    "mary l. trump",
    "mary trump",
    "lara trump",
    "ivanka trump",
    "barron trump",
    "fred trump",

    # Other people named Harris found by the training audit
    "sam harris",
    "simon harris",
    "fred harris",
    "daisy harris",
    "andy harris",
    "liz harris",
    "shawn harris",
    "malcolm harris",
    "laisha harris",
}

### **Compile protected non-candidate names**

The reviewed non-candidate names are converted into a single case-insensitive regular expression so that complete expressions such as `Mary Trump` or `Sam Harris` can be detected before candidate masking is applied.

The pattern is constructed from the `protected_non_candidate_mentions` set rather than written manually. Longer expressions are matched first so that more specific protected names take precedence when expressions overlap.

The boundary checks `(?<!\w)` and `(?!\w)` ensure that only complete expressions are protected rather than matching the same character sequence inside a larger word.

During masking, these expressions are temporarily replaced with internal placeholders, candidate names are masked, and the protected expressions are then restored unchanged.

In [530]:
# Compile all reviewed non-candidate names into one regex. Longer expressions are matched first to handle overlapping names.
protected_non_candidate_pattern = re.compile(
    r"(?<!\w)(?:"
    + "|".join(
        re.escape(phrase)
        for phrase in sorted(
            protected_non_candidate_mentions,
            key=len,
            reverse=True,
        )
    )
    + r")(?!\w)",
    flags=re.IGNORECASE,
)

### **Freeze candidate-masking exceptions**

The final masking rule distinguishes between two types of exceptions.

First, surface-level false positives and ambiguous forms identified during the training-data review are excluded from masking. These include lexical forms such as `mcdonalds`, `harrison`, or `trumpet`.

Second, complete names such as `Mary Trump` or `Sam Harris` are temporarily protected because the individual surname `Trump` or `Harris` would otherwise be correctly recognized as a candidate-name component.

In [531]:
# Combine confirmed false positives from normal variants and hashtags
candidate_false_positives = (normal_false_positives | hashtag_false_positives)


# Combine ambiguous expressions from normal variants and hashtags
candidate_ambiguous = (normal_ambiguous | hashtag_ambiguous)


# Neither false positives nor ambiguous forms will be masked
candidate_masking_exclusions = (candidate_false_positives | candidate_ambiguous)

print(f"False positives: {len(candidate_false_positives):,}")
print(f"Ambiguous forms: {len(candidate_ambiguous):,}")
print(f"Total excluded:  {len(candidate_masking_exclusions):,}")

False positives: 21
Ambiguous forms: 10
Total excluded:  31


### **Construct the candidate-masking function**

The frozen training-derived rules are combined into the final masking function.

The procedure first protects complete references to non-candidate people. It then masks explicit candidate names and candidate-name components occurring inside hashtags, compounds, and other surface forms. Finally, the protected non-candidate names are restored.

Only the candidate-identifying component is replaced whenever possible so that surrounding stance-related information remains available to the model.

In [532]:
# Match complete candidate names first.
full_candidate_name_pattern = re.compile(
    r"\b(?:"
    r"Donald(?:\s+J\.?)?\s+Trump"
    r"|"
    r"Kamala(?:\s+(?:D\.?|Devi))?\s+Harris"
    r")\b",
    flags=re.IGNORECASE,
)


# Match candidate-identifying components within larger surface forms.
candidate_component_pattern = re.compile(
    r"(?:"
    r"Donald[-_]?Trump"
    r"|Kamala[-_]?Harris"
    r"|Trump"
    r"|Donald"
    r"|Harris"
    r"|Kamala"
    r")",
    flags=re.IGNORECASE,
)


def mask_candidate_surface(match):
    """Mask candidate-name components within one surface form."""
    surface = match.group(0)

    # Reviewed false positives and ambiguous forms remain unchanged.
    if surface.lower() in candidate_masking_exclusions:
        return surface

    # Preserve surrounding information and replace only
    # candidate-identifying components.
    return candidate_component_pattern.sub(
        "[CANDIDATE]",
        surface,
    )


def mask_candidate_mentions(text):
    """Remove explicit candidate-name information from one post."""
    if not isinstance(text, str):
        return text

    # 1. Temporarily protect references to other people.
    protected_mentions = {}

    def protect_non_candidate(match):
        placeholder = (
            f"__NONCANDIDATE_PERSON_"
            f"{len(protected_mentions)}__"
        )
        protected_mentions[placeholder] = match.group(0)
        return placeholder

    masked_text = protected_non_candidate_pattern.sub(
        protect_non_candidate,
        text,
    )

    # 2. Replace complete candidate names with one placeholder.
    masked_text = full_candidate_name_pattern.sub(
        "[CANDIDATE]",
        masked_text,
    )

    # 3. Mask remaining isolated names, hashtags,
    # compounds, and related surface forms.
    masked_text = variant_pattern.sub(
        mask_candidate_surface,
        masked_text,
    )

    # 4. Restore protected non-candidate names.
    for placeholder, original_surface in protected_mentions.items():
        masked_text = masked_text.replace(
            placeholder,
            original_surface,
        )

    return masked_text

### **Create the candidate-masked test set**

The frozen masking rule is now applied unchanged to the retrieved context posts of the human-annotated test set.

Only the `Content` field of each context post is modified. The supplied `TargetEntity`, stance label, post order, and remaining post metadata are preserved.

No masking decisions are added or revised based on the held-out test data.

In [533]:
candidate_masked_test = human_test.copy()

candidate_masked_test["ContextPosts"] = (human_test["ContextPosts"].apply(lambda context_posts: [{**post, "Content": mask_candidate_mentions(post.get("Content")),} for post in context_posts]))

Short Sanity Check to see if everything worked:

In [534]:
total_posts = 0
changed_posts = 0

for original_context, masked_context in zip(
    human_test["ContextPosts"],
    candidate_masked_test["ContextPosts"],
):
    for original_post, masked_post in zip(
        original_context,
        masked_context,
    ):
        total_posts += 1

        if (
            original_post.get("Content")
            != masked_post.get("Content")
        ):
            changed_posts += 1


print(f"Total context posts: {total_posts:,}")
print(f"Posts changed:       {changed_posts:,}")
print(
    f"Share changed:       "
    f"{changed_posts / total_posts:.2%}"
)

Total context posts: 6,829
Posts changed:       3,644
Share changed:       53.36%


In [535]:
masked_examples = []

for original_context, masked_context in zip(
    human_test["ContextPosts"],
    candidate_masked_test["ContextPosts"],
):
    for original_post, masked_post in zip(
        original_context,
        masked_context,
    ):
        original_text = original_post.get("Content")
        masked_text = masked_post.get("Content")

        if original_text != masked_text:
            masked_examples.append({
                "original": original_text,
                "masked": masked_text,
            })


masked_examples = pd.DataFrame(masked_examples)

display(
    masked_examples.sample(
        n=min(30, len(masked_examples)),
        random_state=42,
    )
)

,original,masked
415,I would rather live paycheck to paycheck than ...,I would rather live paycheck to paycheck than ...
2927,UFC just showed a Russian Op. on TV.. and the ...,UFC just showed a Russian Op. on TV.. and the ...
3194,Voters who were reluctant to back Harris becau...,Voters who were reluctant to back [CANDIDATE] ...
298,I still say #FuckTrump and #FucktheGOP,I still say #Fuck[CANDIDATE] and #FucktheGOP
1874,Seen in my PA travels today. \n\nWelcome to Tr...,Seen in my PA travels today. \n\nWelcome to [C...
2691,Pitch in to elect Kamala Harris and Democrats ...,Pitch in to elect [CANDIDATE] and Democrats na...
32,Trump getting his new Legion of Doom all toget...,[CANDIDATE] getting his new Legion of Doom all...
3313,Notice how the same folks who called Kamala Ha...,Notice how the same folks who called [CANDIDAT...
2629,It’s time! \n\n#KamalaHarris4President \n\n💙🌊💙...,It’s time! \n\n#[CANDIDATE]4President \n\n💙🌊💙🌊...
2897,"I would like to meet a person who is like ""I w...","I would like to meet a person who is like ""I w..."


### **Document actually masked candidate surface forms**

For reproducibility, the following audit records which candidate-related surface
forms were actually modified by the frozen Candidate-Mention Masking rule in
the human-annotated test set.

This table is created only after the masking rule has been finalized. It is
therefore used for documentation rather than for developing or modifying the
masking rule.

For each modified surface form, the table records its original form, the
resulting masked form, and the number of occurrences in the test contexts.

In [536]:
masked_surface_forms = []

for context_posts in human_test["ContextPosts"]:
    for post in context_posts:
        text = post.get("Content")

        if not isinstance(text, str):
            continue

        # Temporarily protect non-candidate people exactly as in mask_candidate_mentions().
        protected_mentions = {}

        def protect_non_candidate_for_audit(match):
            placeholder = (
                f"__NONCANDIDATE_PERSON_"
                f"{len(protected_mentions)}__"
            )
            protected_mentions[placeholder] = match.group(0)
            return placeholder

        audit_text = protected_non_candidate_pattern.sub(
            protect_non_candidate_for_audit,
            text,
        )

        # Complete candidate names
        for match in full_candidate_name_pattern.finditer(audit_text):
            original_surface = match.group(0)

            masked_surface_forms.append({
                "original_surface": original_surface,
                "masked_surface": "[CANDIDATE]",
            })

        # Remove complete names temporarily so that their components are not counted a second time.
        audit_text = full_candidate_name_pattern.sub(
            "[CANDIDATE]",
            audit_text,
        )

        # Remaining variants / hashtags / compounds
        for match in variant_pattern.finditer(audit_text):
            original_surface = match.group(0)
            masked_surface = mask_candidate_surface(match)

            if original_surface != masked_surface:
                masked_surface_forms.append({
                    "original_surface": original_surface,
                    "masked_surface": masked_surface,
                })


candidate_masking_audit = (
    pd.DataFrame(masked_surface_forms)
    .groupby(
        ["original_surface", "masked_surface"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "count"})
    .sort_values(
        ["count", "original_surface"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

In [537]:
candidate_masking_audit.to_csv("../data/interventions/" "candidate_mention_masking_audit.csv", index=False)

The audit confirms that the rule generalizes beyond exact forms observed during
training, including previously unseen hashtags and compounds containing candidate
name components. Because matching is intentionally recall-oriented, rare
non-candidate expressions containing the same character sequences may also be
modified.

### **Save the candidate-masked test set**

The completed intervention is stored separately from the original human-annotated test set for later model evaluation.

In [538]:
candidate_masked_test.to_parquet("../data/interventions/" "human_test_candidate_mentions_masked.parquet", index=False)

---

## **Matched Random Control for Candidate-Mention Masking**

To distinguish the effect of removing explicit candidate information from the
general effect of perturbing text, Candidate-Mention Masking is accompanied by
a matched random-removal control.

For each context post, the control determines how many word-token positions are
affected by the frozen Candidate-Mention Masking rule. It then randomly selects
the same number of eligible non-candidate word-token positions from that same
post and replaces them with `[REMOVED]`.

Matching is performed separately within each post. This preserves which posts
in the retrieved history are perturbed and controls for the amount and location
of removed lexical information as closely as possible.

Explicit candidate references, reviewed protected non-candidate names, and URLs
are excluded from the pool of possible random-control positions.

For very short posts, there may be fewer eligible alternative tokens than
candidate-affected token positions. In these cases, all available eligible
tokens are removed and the remaining shortfall is recorded rather than
transferring the perturbation to another context post.

Five independently seeded control datasets are created to reduce dependence on
one particular random draw.

The control is matched on the number of affected word-token positions, not on
the exact surface transformation. Candidate-Mention Masking may replace a
multi-token candidate name such as `Donald Trump` with one `[CANDIDATE]`
placeholder, while the control independently masks the corresponding number of
eligible word-token positions.

In [539]:
candidate_control_placeholder = "[REMOVED]"

# Word-token positions used to quantify the amount of candidate information affected by Intervention 3.
candidate_control_token_pattern = re.compile(r"(?u)\b\w+\b")


# URLs and bare domain names are excluded from random control masking.
url_pattern = re.compile(
    r"(?:https?://|www\.)\S+"
    r"|(?<![\w@])(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+"
    r"[a-z]{2,}(?:/[^\s]*)?",
    flags=re.IGNORECASE,
)


def spans_overlap(a, b):
    """Return True if two character spans overlap."""
    return a[0] < b[1] and b[0] < a[1]


def get_candidate_reference_char_spans(text):
    """
    Return character spans treated as candidate references by
    the frozen Intervention-3 masking rule.
    """

    if not isinstance(text, str):
        return []

    candidate_spans = []

    # Complete non-candidate names such as Mary Trump or Sam Harris
    # are protected from candidate-reference detection.
    protected_spans = [
        match.span()
        for match in protected_non_candidate_pattern.finditer(text)
    ]

    # Complete candidate names
    for match in full_candidate_name_pattern.finditer(text):
        span = match.span()

        if not any(
            spans_overlap(span, protected_span)
            for protected_span in protected_spans
        ):
            candidate_spans.append(span)

    # Remaining isolated names, hashtags, compounds, and variants
    for match in variant_pattern.finditer(text):
        span = match.span()

        if any(
            spans_overlap(span, protected_span)
            for protected_span in protected_spans
        ):
            continue

        surface = match.group(0)

        if mask_candidate_surface(match) == surface:
            continue

        # Record only the candidate-identifying component,
        # not the complete surrounding surface form.
        for component_match in candidate_component_pattern.finditer(surface):
            candidate_spans.append(
                (
                    match.start() + component_match.start(),
                    match.start() + component_match.end(),
                )
            )

    return candidate_spans

In [540]:
def get_candidate_control_info(text):
    """
    Identify candidate-affected word-token positions and eligible
    matched-control positions in one post.
    """

    if not isinstance(text, str):
        return [], set(), []

    token_matches = list(
        candidate_control_token_pattern.finditer(text)
    )

    candidate_char_spans = (
        get_candidate_reference_char_spans(text)
    )

    # All word-token positions overlapping an actual candidate reference
    candidate_indices = {
        token_idx
        for token_idx, token_match in enumerate(token_matches)
        if any(
            spans_overlap(
                token_match.span(),
                candidate_span,
            )
            for candidate_span in candidate_char_spans
        )
    }

    # Random controls must not remove candidate references themselves,
    # URLs, or explicitly protected non-candidate names.
    protected_control_spans = (
        candidate_char_spans
        + [
            match.span()
            for match in url_pattern.finditer(text)
        ]
        + [
            match.span()
            for match in protected_non_candidate_pattern.finditer(text)
        ]
    )

    eligible_indices = []

    for token_idx, token_match in enumerate(token_matches):

        if token_idx in candidate_indices:
            continue

        if any(
            spans_overlap(
                token_match.span(),
                protected_span,
            )
            for protected_span in protected_control_spans
        ):
            continue

        eligible_indices.append(token_idx)

    return (
        token_matches,
        candidate_indices,
        eligible_indices,
    )

In [541]:
def apply_candidate_control_mask(
    text,
    token_matches,
    selected_token_indices,
):
    """
    Replace selected word-token positions with the neutral
    candidate-control placeholder.
    """

    if not isinstance(text, str):
        return text

    if not selected_token_indices:
        return text

    char_spans = [
        token_matches[int(token_idx)].span()
        for token_idx in selected_token_indices
    ]

    masked_text = text

    # Replace right-to-left so original character offsets remain valid.
    for start, end in sorted(
        char_spans,
        reverse=True,
    ):
        masked_text = (
            masked_text[:start]
            + candidate_control_placeholder
            + masked_text[end:]
        )

    return masked_text

In [542]:
def create_candidate_matched_random_control(df, seed):
    """
    Create one post-level matched random-control version of
    Candidate-Mention Masking.

    For each post, the control removes as many eligible
    non-candidate word-token positions as Candidate-Mention
    Masking affects in that same post.

    If a post contains too few eligible alternative tokens,
    the remaining shortfall is retained and reported rather
    than being transferred to another post.
    """

    rng = np.random.default_rng(seed)

    control_df = df.copy(deep=True)

    new_contexts = []
    audit_rows = []

    for example_idx, row in enumerate(
        df.itertuples(index=False)
    ):

        masked_context = []

        for post_idx, post in enumerate(row.ContextPosts):

            masked_post = post.copy()
            text = post.get("Content")

            (
                token_matches,
                candidate_indices,
                eligible_indices,
            ) = get_candidate_control_info(text)

            requested_tokens = len(candidate_indices)

            n_remove = min(
                requested_tokens,
                len(eligible_indices),
            )

            if n_remove > 0:
                selected_indices = rng.choice(
                    eligible_indices,
                    size=n_remove,
                    replace=False,
                )

                selected_indices = {
                    int(token_idx)
                    for token_idx in selected_indices
                }

            else:
                selected_indices = set()

            masked_post["Content"] = (
                apply_candidate_control_mask(
                    text,
                    token_matches,
                    selected_indices,
                )
            )

            masked_context.append(masked_post)

            audit_rows.append({
                "example": example_idx,
                "post": post_idx,
                "target": row.TargetEntity,
                "requested_tokens": requested_tokens,
                "available_control_tokens": len(
                    eligible_indices
                ),
                "removed_tokens": n_remove,
                "shortfall_tokens": (
                    requested_tokens - n_remove
                ),
                "exact_match": (
                    requested_tokens == n_remove
                ),
            })

        new_contexts.append(masked_context)

    control_df["ContextPosts"] = new_contexts

    audit_df = pd.DataFrame(audit_rows)

    return control_df, audit_df

In [543]:
candidate_control_seeds = [1, 2, 3, 4, 5]

candidate_controls = {}
candidate_control_audits = {}

for seed in candidate_control_seeds:

    (
        candidate_controls[seed],
        candidate_control_audits[seed],
    ) = create_candidate_matched_random_control(
        human_test,
        seed=seed,
    )

### **Validate the candidate-matched random controls**

The following audit checks how closely the random-control condition matches
Candidate-Mention Masking at the post level.

Because random removals are restricted to eligible non-candidate tokens from
the same post, exact matching is not always possible for very short posts.
The audit therefore reports both the total token-level matching rate and the
number of post-level shortfalls.

In [544]:
candidate_control_summary = []

for seed in candidate_control_seeds:

    audit = candidate_control_audits[seed]

    affected = audit[
        audit["requested_tokens"] > 0
    ]

    requested = affected[
        "requested_tokens"
    ].sum()

    removed = affected[
        "removed_tokens"
    ].sum()

    candidate_control_summary.append({
        "seed": seed,
        "affected_posts": len(affected),
        "requested_tokens": requested,
        "removed_tokens": removed,
        "shortfall_tokens": affected[
            "shortfall_tokens"
        ].sum(),
        "token_match_%": (
            removed / requested * 100
            if requested > 0
            else 100.0
        ),
        "exact_post_match_%": (
            affected["exact_match"].mean() * 100
            if len(affected) > 0
            else 100.0
        ),
    })


candidate_control_summary = pd.DataFrame(
    candidate_control_summary
)

display(candidate_control_summary)

,seed,affected_posts,requested_tokens,removed_tokens,shortfall_tokens,token_match_%,exact_post_match_%
0,1,3644,5710,5653,57,99.001751,98.682766
1,2,3644,5710,5653,57,99.001751,98.682766
2,3,3644,5710,5653,57,99.001751,98.682766
3,4,3644,5710,5653,57,99.001751,98.682766
4,5,3644,5710,5653,57,99.001751,98.682766


In [545]:
seed = 1

shortfall_posts = (
    candidate_control_audits[seed]
    .query("shortfall_tokens > 0")
)

display(shortfall_posts)

,example,post,target,requested_tokens,available_control_tokens,removed_tokens,shortfall_tokens,exact_match
44,5,3,Trump,2,1,1,1,False
81,10,1,Trump,1,0,0,1,False
127,16,2,Trump,1,0,0,1,False
130,16,5,Trump,2,1,1,1,False
181,23,4,Trump,1,0,0,1,False
232,31,0,Trump,1,0,0,1,False
233,31,1,Trump,1,0,0,1,False
284,38,0,Trump,1,0,0,1,False
315,42,1,Trump,1,0,0,1,False
366,49,1,Trump,1,0,0,1,False


The matched control randomly removed non-candidate word tokens from the same context post as the corresponding candidate reference. Exact matching was possible for approximately 99% of candidate-affected token positions. The small remaining shortfall occurred in very short posts without sufficient eligible alternative tokens; these removals were not transferred to other posts in order to preserve post-level locality.

### **Save as Parquet/Csv**

In [546]:
candidate_control_audit_all = pd.concat(
    [
        audit.assign(seed=seed)
        for seed, audit in candidate_control_audits.items()
    ],
    ignore_index=True,
)

candidate_control_audit_all.to_csv(
    "../data/interventions/"
    "candidate_mention_control_audit.csv",
    index=False,
)

In [547]:
for seed, control_df in candidate_controls.items():

    control_df.to_parquet(
        "../data/interventions/"
        f"human_test_candidate_mentions_control_seed{seed}.parquet",
        index=False,
    )

---

## **Intervention 4: Lexical-Cue Masking**

The fourth intervention removes lexical expressions that are disproportionately
associated with particular stance labels.

Lexical cues are derived exclusively from the training set and separately for
each target and stance label:

- Trump × Favor
- Trump × Against
- Trump × Neither
- Harris × Favor
- Harris × Against
- Harris × Neither

For cue discovery, all retrieved context posts of one user-target pair are
treated as one document. Unigrams and bigrams are represented by binary document
occurrence rather than raw frequency, so repeated use of the same expression by
one user does not increase its contribution.

Candidate-name references are excluded from the resulting feature vocabulary
using the frozen Candidate-Mention Masking rule from Intervention 3.

Association strength is measured using smoothed class-vs-rest log-odds within
each target.

The following parameters define the preprocessing and smoothing settings used for the lexical comparison.

- `min_documents = 10` excludes terms that occur in fewer than 10 user-target documents for the respective target. Very rare terms are more likely to reflect individual examples or noise rather than systematic lexical differences.
- `min_class_df = 10` requires a lexical cue to occur in at least 10 user-target documents of the focal stance class before it can be selected.
- `log_odds_alpha = 0.5` adds a small pseudo-count when calculating log-odds. This prevents problems with zero counts and reduces extreme scores caused by very rare terms.
- `post_boundary_token` marks boundaries between individual posts when texts are combined. This prevents words from separate posts from being treated as if they occurred next to each other.

In [548]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Minimum number of user-target documents in which a lexical feature must occur
min_documents = 10

# Minimum number of user-target documents within the focal stance class in which a lexical feature must occur
min_class_df = 10

# Smoothing parameter used in the log-odds calculation
log_odds_alpha = 0.5

# Special token used to preserve boundaries between concatenated posts
post_boundary_token = "postboundarytoken"

### **Construction of user-target context documents**

For the lexical cue analysis, the retrieved context posts associated with each user-target pair are combined into a single document.

Only non-empty textual post contents are retained. Individual posts are separated by a dedicated boundary token rather than being concatenated directly. This prevents the later n-gram extraction from creating artificial bigrams between the final word of one post and the first word of the next.

In [549]:
def build_user_target_document(context_posts):
    """Concatenate retrieved context posts while preserving post boundaries."""

    # Keep only non-empty textual post contents
    texts = [
        post.get("Content")
        for post in context_posts
        if isinstance(post.get("Content"), str)
        and post.get("Content").strip()
    ]

    # Join posts using a dedicated boundary token so that later n-gram extraction does not create artificial bigrams across posts
    return f" {post_boundary_token} ".join(texts)


# Create a training table containing the user-target pair, stance label, and corresponding lexical context document
cue_train = train[["UserId", "TargetEntity", "StanceLabel"]].copy()

# Combine all retrieved context posts for each user-target pair into one document for lexical feature extraction
cue_train["CueDocument"] = (train["ContextPosts"].apply(build_user_target_document))

cue_train.head()

,UserId,TargetEntity,StanceLabel,CueDocument
0,9,Harris,Favor,"I don't see how you read ""We are withholding o..."
1,9,Trump,Against,"I don't see how you read ""We are withholding o..."
2,10,Harris,Favor,Kamala Harris is very good at this. postbounda...
3,10,Trump,Against,AP still has Trump up postboundarytoken Not to...
4,49,Harris,Against,Let be real @anonymous\n\nJoe Scarborough is N...


### **Candidate-reference detection**

A phrase is treated as containing a candidate reference if applying the frozen masking rule changes the phrase.  
This reuses the same masking logic defined for Intervention 3 rather than introducing a separate detection rule.

In [550]:
def contains_candidate_reference(phrase):
    """Return True if the frozen Intervention-3 rule modifies the phrase."""

    return mask_candidate_mentions(phrase) != phrase

### **Construction of the lexical document-term matrix**

The combined context documents are transformed into a binary document-term matrix using `CountVectorizer`.

Both unigrams and bigrams are considered. A feature records whether a phrase occurs in a document, rather than its within-document frequency. Terms occurring in fewer than `min_documents` documents are excluded to reduce noise from very rare lexical features.

The function returns the fitted vectorizer, the resulting document-term matrix, and the corresponding unigram and bigram labels.

In [551]:
def build_target_document_matrix(target_data):

    # Initialize a vectorizer that extracts unigrams and bigrams occurring in at least the required number of documents
    vectorizer = CountVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=min_documents,
        binary=True,
    )

    # Learn the vocabulary from the cue documents and transform each document into a binary feature vector
    X = vectorizer.fit_transform(
        target_data["CueDocument"]
    )

    # Retrieve the corresponding unigram and bigram names
    phrases = vectorizer.get_feature_names_out()

    return vectorizer, X, phrases

### **Filtering lexical features**

Before lexical cues are analyzed, features that should not be treated as ordinary lexical information are removed.

Phrases containing the artificial post-boundary token are excluded because this token exists only to preserve post boundaries during preprocessing. Explicit candidate references are also excluded, since candidate-identifying information is handled separately by the candidate-masking intervention.

All remaining unigrams and bigrams are retained as eligible lexical features.

In [552]:
def lexical_feature_is_allowed(phrase):

    # Split the unigram or bigram into individual tokens
    tokens = phrase.split()

    # Exclude the artificial token used to separate individual posts
    if post_boundary_token in tokens:
        return False

    # Explicit candidate references belong to Intervention 3, not Lexical-Cue Masking.
    if contains_candidate_reference(phrase):
        return False

    # Keep all remaining lexical features
    return True

### **Stance-specific lexical cue scoring**

For each target, lexical features are evaluated separately for the `Favor`, `Against`, and `Neither` stance classes.

For a given stance, the document frequency of each unigram or bigram is compared with its document frequency across all remaining stance classes. A smoothed log-odds score is used to quantify this association. Positive scores indicate that a phrase occurs proportionally more often in the current stance class.

Candidate-identifying expressions and technical post-boundary tokens are excluded beforehand. The remaining positively associated lexical cues are ranked primarily by log-odds score, with document frequency used as a secondary criterion.

In [553]:
def score_lexical_cues_for_target(target_data):

    # All rows in target_data belong to the same target
    target = target_data["TargetEntity"].iloc[0]

    # Build the binary document-term matrix
    _, X, phrases = build_target_document_matrix(target_data)

    # Remove candidate-related and technical features.
    allowed = np.array([lexical_feature_is_allowed(phrase) for phrase in phrases])

    X = X[:, allowed]
    phrases = phrases[allowed]

    result_tables = []

    labels = ["Favor", "Against", "Neither"]

    # Compare each stance class against all remaining classes
    for label in labels:

        class_mask = (target_data["StanceLabel"].eq(label).to_numpy())

        n_class = class_mask.sum()
        n_other = (~class_mask).sum()

        # Because X is binary, summing gives document frequency.
        df_class = np.asarray(X[class_mask].sum(axis=0)).ravel()

        df_other = np.asarray(X[~class_mask].sum(axis=0)).ravel()

        log_odds = (np.log((df_class + log_odds_alpha)/ (n_class - df_class + log_odds_alpha))
            - np.log((df_other + log_odds_alpha) / (n_other - df_other + log_odds_alpha)))

        scores = pd.DataFrame({
            "target": target,
            "label": label,
            "phrase": phrases,
            "ngram": [
                len(phrase.split())
                for phrase in phrases
            ],
            "df_class": df_class,
            "df_other": df_other,
            "df_total": df_class + df_other,
            "log_odds": log_odds,
        })

        # We only want expressions positively associated
        # with this stance class.
        scores = (
            scores[
                (scores["log_odds"] > 0)
                & (scores["df_class"] >= min_class_df)
            ]
            .sort_values(
                ["log_odds", "df_class", "phrase"],
                ascending=[False, False, True],
            )
            .reset_index(drop=True)
        )

        scores["rank"] = (np.arange(len(scores)) + 1)

        result_tables.append(scores)

    return pd.concat(result_tables, ignore_index=True,)

In [554]:
# Calculate stance-specific lexical cue scores separately for each target
lexical_cue_scores = pd.concat(
    [score_lexical_cues_for_target(target_data.reset_index(drop=True))
        for _, target_data in cue_train.groupby("TargetEntity", sort=True)
    ], ignore_index=True)

In [555]:
# Keep only the 25 highest-ranked lexical cues for each target and stance class
top25_lexical_cues = (
    lexical_cue_scores[
        lexical_cue_scores["rank"] <= 25
    ]
    .copy()
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(top25_lexical_cues[["target", "label", "phrase", "ngram", "df_class", "df_other", "df_total", "log_odds", "rank",]])

,target,label,phrase,ngram,df_class,df_other,df_total,log_odds,rank
0,Harris,Favor,felons and,2,74,0,74,5.719284,1
1,Harris,Favor,way my,2,73,0,73,5.705290,2
2,Harris,Favor,against adjudicated,2,72,0,72,5.691112,3
3,Harris,Favor,and insurrection,2,72,0,72,5.691112,4
4,Harris,Favor,inciting racists,2,72,0,72,5.691112,5
5,Harris,Favor,insurrection inciting,2,72,0,72,5.691112,6
6,Harris,Favor,racists always,2,72,0,72,5.691112,7
7,Harris,Favor,rapists convicted,2,72,0,72,5.691112,8
8,Harris,Favor,vote cast,2,72,0,72,5.691112,9
9,Harris,Favor,about dem,2,37,0,37,5.015223,10


### **Manual audit of selected lexical cues**

Selected lexical cues can be traced back to the individual training posts in which they occur.

To ensure consistency with the lexical feature extraction, the audit uses the same lowercase tokenization rule as `CountVectorizer`. A cue is considered present only when its tokens occur as an exact consecutive sequence within a single context post.

The function restricts the search to a specified target and stance class and returns the corresponding user IDs and original post texts for manual inspection.

In [556]:
cue_token_pattern = re.compile(r"(?u)\b\w\w+\b")

def tokenize_for_cue_audit(text):
    """Replicate CountVectorizer's default lowercase tokenization."""
    if not isinstance(text, str):
        return []

    # Convert text to lowercase and extract all valid tokens
    return cue_token_pattern.findall(text.lower())


def audit_lexical_cue(target, label, phrase):
    """Inspect training posts containing a selected lexical cue exactly."""

    # Restrict the training data to the selected target and stance class
    subset = train[
        (train["TargetEntity"] == target)
        & (train["StanceLabel"] == label)
    ]

    # Tokenize the lexical cue using the same rule as the post text
    phrase_tokens = tokenize_for_cue_audit(phrase)
    n = len(phrase_tokens)

    matches = []

    # Inspect all context posts belonging to the selected examples
    for row in subset.itertuples():
        for post in row.ContextPosts:

            text = post.get("Content")

            if not isinstance(text, str):
                continue

            text_tokens = tokenize_for_cue_audit(text)

            contains_cue = any(
                text_tokens[i:i + n] == phrase_tokens
                for i in range(len(text_tokens) - n + 1)
            )

            if contains_cue:
                matches.append({
                    "UserId": row.UserId,
                    "Content": text,
                })

    return pd.DataFrame(matches)

In [557]:
audit_for_as = audit_lexical_cue(target="Trump", label="Against", phrase="for as")

print("Users containing cue:", audit_for_as["UserId"].nunique())

print("Distinct exact post texts:", audit_for_as["Content"].nunique())

display(audit_for_as["Content"].value_counts().head(10))

Users containing cue: 215
Distinct exact post texts: 10


Content
For as long as I live, I will never understand how anyone could have chosen Trump over this.                                                                                                                                                                                              184
For as long as I live, I will never forget the cowardice of the people who let Donald Trump destroy America.                                                                                                                                                                               13
"The plain fact is that Donald Trump is not just a bad man. He is an avatar for iniquity and immorality and selfishness." We've known this for as long as Trump has been on the political scene. From the archives:                                                                         9
For as long as I live, I will never forget how the DOJ let Elon Musk give Donald Trump another presidency.                            

In [558]:
audit_felons_and = audit_lexical_cue(target="Harris", label="Favor", phrase="felons and",)

print("Users containing cue:", audit_felons_and["UserId"].nunique())

print("Distinct exact post texts:", audit_felons_and["Content"].nunique())

display(audit_felons_and["Content"].value_counts().head(10))

Users containing cue: 74
Distinct exact post texts: 3


Content
I will ALWAYS be proud of the vote I cast for VP Kamala Harris for President of the United States. \n\nShe IS better than he is, in every way.\nMY vote was on the right side of history.\n\nI will always vote against adjudicated rapists, convicted felons, and insurrection inciting racists.\n\nALWAYS.    72
#Texas #HillCountry #HarrisWalz supporters regrouping for #2026 #FuckPutin #FuckTrump we will take back our country from the Oligarchs, convicted felons, and Russian agents ✊🏽                                                                                                                                  1
If we ever have an election again, I'm voting for the person who guarantees not to hand the country over to fascists, felons, and terrorists.                                                                                                                                                                    1
Name: count, dtype: int64

### **Audit of highly associated lexical cues**

Several of the highest-ranked lexical cues were found to originate from
identical post texts occurring across multiple user-target documents. For
example, 72 of 74 Harris-Favor documents containing `felons and` contained the
same exact post, while 184 of 215 Trump-Against documents containing for as shared the same exact post text.

These repetitions are retained rather than deduplicated because they are part
of the input distribution available to the models and may themselves constitute
dataset-specific lexical shortcuts. The audit is therefore descriptive and does
not alter the cue-ranking procedure.

### **Save top 10/25 Lexical cues as CSV**:

In [559]:
lexical_cues_top10 = (lexical_cue_scores[lexical_cue_scores["rank"] <= 10].copy())

lexical_cues_top25 = (lexical_cue_scores[lexical_cue_scores["rank"] <= 25].copy())

In [560]:
lexical_cue_scores.to_csv("../data/interventions/lexical_cue_scores.csv", index=False)

lexical_cues_top10.to_csv("../data/interventions/lexical_cues_top10.csv",index=False)

lexical_cues_top25.to_csv("../data/interventions/lexical_cues_top25.csv", index=False)

### **Target-specific lexical cue sets**

The selected lexical cues are grouped by target and converted into sets of unique phrases.

This creates one cue set per target, independent of stance class, which can later be used to check whether a post contains any of the selected lexical cues.

In [561]:
def build_target_cue_sets(cue_table):
    """Combine Favor, Against, and Neither cues for each target."""

     # Group the cue table by target and convert the phrases for each target into a set of unique lexical cues
    return {target: set(group["phrase"]) for target, group in cue_table.groupby("target")}


top10_cues_by_target = build_target_cue_sets(lexical_cues_top10)

top25_cues_by_target = build_target_cue_sets(lexical_cues_top25)

### **Token-consistent matching of selected lexical cues**

The selected lexical cues are converted into token sequences using the same
tokenization rule as the preceding `CountVectorizer`-based cue extraction and
manual audit.

This is necessary because `CountVectorizer` constructs n-grams from consecutive
tokens rather than from literal whitespace-separated strings. For example,
`felons and` is also extracted from `felons, and`, despite the intervening
punctuation.

The masking procedure therefore identifies cue occurrences at the token level. All word-token positions belonging to at least one selected cue are collected, and each affected token position is replaced exactly once with [REMOVED]. This also handles overlapping lexical cues without masking the same token position multiple times.

In [562]:
lexical_placeholder = "[REMOVED]"


def build_cue_sequences(cues):
    """Convert selected lexical cues into CountVectorizer-compatible token sequences."""

    sequences = {
        tuple(tokenize_for_cue_audit(cue))
        for cue in cues
    }

    # Remove possible empty token sequences
    sequences.discard(())

    # Check bigrams before unigrams
    return sorted(
        sequences,
        key=lambda sequence: (-len(sequence), sequence),
    )

In [563]:
top10_cue_sequences_by_target = {target: build_cue_sequences(cues) for target, cues in top10_cues_by_target.items()}

top25_cue_sequences_by_target = {target: build_cue_sequences(cues) for target, cues in top25_cues_by_target.items()}

In [564]:
top10_cue_sequences_by_target

{'Harris': [('about', 'dem'),
  ('against', 'adjudicated'),
  ('and', 'insurrection'),
  ('as', 'nancy'),
  ('center', 'right'),
  ('does', 'she'),
  ('felons', 'and'),
  ('for', 'genocide'),
  ('gaetz', 'says'),
  ('genocide', 'in'),
  ('house', 'ethics'),
  ('inciting', 'racists'),
  ('insurrection', 'inciting'),
  ('lost', 'and'),
  ('name', 'from'),
  ('on', 'trans'),
  ('racists', 'always'),
  ('rapists', 'convicted'),
  ('the', 'genocide'),
  ('vote', 'cast'),
  ('way', 'my'),
  ('yelling', 'at'),
  ('bhattacharya',),
  ('carr',),
  ('commission',),
  ('eu',),
  ('genocidal',),
  ('internal',),
  ('merkel',),
  ('taps',)],
 'Trump': [('america', 'great'),
  ('anyone', 'could'),
  ('as', 'live'),
  ('destroy', 'them'),
  ('echo', 'chamber'),
  ('for', 'as'),
  ('former', 'florida'),
  ('great', 'again'),
  ('have', 'chosen'),
  ('how', 'anyone'),
  ('live', 'will'),
  ('members', 'to'),
  ('once', 'you'),
  ('presidential', 'election'),
  ('the', 'latest'),
  ('the', 'left'),
  ('

### **Application of target-specific lexical-cue masking**

The previously selected lexical cues are applied to the held-out test set without modifying the original data.

For each user-target example, the masking pattern corresponding to the target is selected and applied separately to every post in `ContextPosts`. Matching lexical cues are replaced with the generic `[REMOVED]` placeholder, while all remaining post content and metadata are preserved.

Two intervention datasets are created. The first uses the union of the top-10 cues selected separately for each stance class within each target, while the second analogously uses the top-25 cues. The resulting intervention cue sets are target-specific but independent of the test example's gold stance label.

In [565]:
def mask_lexical_cues_in_text(text, cue_sequences):
    """Mask every word-token position belonging to a selected lexical cue."""

    if not isinstance(text, str):
        return text

    # Locate tokens using the same tokenization as cue extraction
    token_matches = list(cue_token_pattern.finditer(text))

    tokens = [
        match.group(0).lower()
        for match in token_matches
    ]

    cue_indices = set()

    # Identify every token position that belongs to at least one selected cue
    for i in range(len(tokens)):
        for cue_sequence in cue_sequences:
            n = len(cue_sequence)

            if tuple(tokens[i:i + n]) == cue_sequence:
                cue_indices.update(range(i, i + n))

    if not cue_indices:
        return text

    # Replace affected tokens from right to left so character offsets remain valid
    masked_text = text

    for token_idx in sorted(cue_indices, reverse=True):
        start, end = token_matches[token_idx].span()

        masked_text = (
            masked_text[:start]
            + lexical_placeholder
            + masked_text[end:]
        )

    return masked_text

In [566]:
def mask_lexical_cues_in_context(
    context_posts,
    target,
    cue_sequences_by_target,
):
    """Mask selected target-specific lexical cues in ContextPosts."""

    cue_sequences = cue_sequences_by_target[target]

    masked_posts = []

    for post in context_posts:

        masked_post = post.copy()

        masked_post["Content"] = mask_lexical_cues_in_text(
            post.get("Content"),
            cue_sequences,
        )

        masked_posts.append(masked_post)

    return masked_posts

In [567]:
# Create independent copies of the held-out test set
lexical_top10_masked_test = human_test.copy(deep=True)

lexical_top25_masked_test = human_test.copy(deep=True)


# Apply target-specific top-10 lexical-cue masking
lexical_top10_masked_test["ContextPosts"] = [
    mask_lexical_cues_in_context(
        context_posts=row.ContextPosts,
        target=row.TargetEntity,
        cue_sequences_by_target=top10_cue_sequences_by_target,
    )
    for row in human_test.itertuples()
]


# Apply target-specific top-25 lexical-cue masking
lexical_top25_masked_test["ContextPosts"] = [
    mask_lexical_cues_in_context(
        context_posts=row.ContextPosts,
        target=row.TargetEntity,
        cue_sequences_by_target=top25_cue_sequences_by_target,
    )
    for row in human_test.itertuples()
]

In [568]:
def summarize_lexical_masking(original_df, masked_df, variant):
    """Summarize how much text was affected by lexical-cue masking."""

    total_posts = 0
    masked_posts = 0
    masked_examples = 0
    total_masks = 0

    for original_row, masked_row in zip(
        original_df.itertuples(),
        masked_df.itertuples(),
    ):
        example_masks = 0

        for original_post, masked_post in zip(
            original_row.ContextPosts,
            masked_row.ContextPosts,
        ):
            total_posts += 1

            original_text = original_post.get("Content")
            masked_text = masked_post.get("Content")

            if not isinstance(original_text, str) or not isinstance(masked_text, str):
                continue

            # Count placeholders newly introduced by this intervention
            n_masks = (
                masked_text.count(lexical_placeholder)
                - original_text.count(lexical_placeholder)
            )

            if n_masks > 0:
                masked_posts += 1
                example_masks += n_masks
                total_masks += n_masks

        if example_masks > 0:
            masked_examples += 1

    return {
        "variant": variant,
        "examples": len(original_df),
        "masked_examples": masked_examples,
        "masked_examples_%": round(
            100 * masked_examples / len(original_df), 2
        ),
        "posts": total_posts,
        "masked_posts": masked_posts,
        "masked_posts_%": round(
            100 * masked_posts / total_posts, 2
        ),
        "total_masks": total_masks,
    }


masking_summary = pd.DataFrame([
    summarize_lexical_masking(
        human_test,
        lexical_top10_masked_test,
        "Top 10",
    ),
    summarize_lexical_masking(
        human_test,
        lexical_top25_masked_test,
        "Top 25",
    ),
])

display(masking_summary)

,variant,examples,masked_examples,masked_examples_%,posts,masked_posts,masked_posts_%,total_masks
0,Top 10,890,124,13.93,6829,150,2.20,703
1,Top 25,890,265,29.78,6829,358,5.24,1117


In [569]:
def summarize_masking_by_target(original_df, masked_df, variant):
    rows = []

    for target in original_df["TargetEntity"].unique():
        mask = original_df["TargetEntity"] == target

        result = summarize_lexical_masking(
            original_df.loc[mask],
            masked_df.loc[mask],
            variant,
        )

        result["target"] = target
        rows.append(result)

    return rows


masking_by_target = pd.DataFrame(
    summarize_masking_by_target(
        human_test,
        lexical_top10_masked_test,
        "Top 10",
    )
    +
    summarize_masking_by_target(
        human_test,
        lexical_top25_masked_test,
        "Top 25",
    )
)

display(masking_by_target[["variant", "target", "examples","masked_examples", "masked_examples_%", "masked_posts_%", "total_masks",]])

,variant,target,examples,masked_examples,masked_examples_%,masked_posts_%,total_masks
0,Top 10,Trump,445,76,17.08,2.85,403
1,Top 10,Harris,445,48,10.79,1.58,300
2,Top 25,Trump,445,181,40.67,7.49,688
3,Top 25,Harris,445,84,18.88,3.09,429


### **Save the final Parquet**

In [570]:
lexical_top10_masked_test.to_parquet("../data/interventions/" "human_test_lexical_cues_top10_masked.parquet", index=False)

lexical_top25_masked_test.to_parquet("../data/interventions/" "human_test_lexical_cues_top25_masked.parquet", index=False)

---

## **Control condition for Intervention 4: Matched Random Masking**

To distinguish the specific effect of removing stance-associated lexical cues from the general effect of deleting textual information, matched random control variants are created for the top-10 and top-25 lexical-cue interventions.

Matching is performed separately for each user-target example across all of its context posts. If lexical-cue masking affects `n` word-token positions in an example, the corresponding control condition randomly masks the same number of eligible word-token positions whenever possible.

Tokens belonging to selected lexical-cue occurrences, explicit candidate references, or URLs are excluded from random sampling. Examples without lexical-cue occurrences remain unchanged.

Five control variants with fixed random seeds are generated for each lexical-cue condition to reduce dependence on a particular random sample.

In [ ]:
def get_candidate_reference_char_spans(text):
    """Return character spans that Intervention 3 treats as candidate references."""

    if not isinstance(text, str):
        return []

    candidate_spans = []

    # References to other people such as Mary Trump remain unprotected
    # because Intervention 3 deliberately preserves them.
    protected_spans = [
        match.span()
        for match in protected_non_candidate_pattern.finditer(text)
    ]

    # Complete candidate names
    for match in full_candidate_name_pattern.finditer(text):
        span = match.span()

        if not any(
            spans_overlap(span, protected_span)
            for protected_span in protected_spans
        ):
            candidate_spans.append(span)

    # Remaining candidate-related variants
    for match in variant_pattern.finditer(text):
        span = match.span()

        if any(
            spans_overlap(span, protected_span)
            for protected_span in protected_spans
        ):
            continue

        if mask_candidate_surface(match) != match.group(0):
            candidate_spans.append(span)

    return candidate_spans


def get_control_info(text, cue_sequences):
    """
    Identify lexical-cue tokens and eligible control tokens in one post.
    """

    if not isinstance(text, str):
        return [], set(), []

    token_matches = list(cue_token_pattern.finditer(text))
    tokens = [
        match.group(0).lower()
        for match in token_matches
    ]

    cue_indices = set()

    # Identify all token positions affected by lexical-cue masking
    for i in range(len(tokens)):
        for cue_sequence in cue_sequences:
            n = len(cue_sequence)

            if tuple(tokens[i:i + n]) == cue_sequence:
                cue_indices.update(range(i, i + n))

    # Candidate references and URLs cannot be random controls
    protected_char_spans = (
        get_candidate_reference_char_spans(text)
        + [
            match.span()
            for match in url_pattern.finditer(text)
        ]
    )

    eligible_indices = []

    for i, token_match in enumerate(token_matches):

        if i in cue_indices:
            continue

        if any(
            spans_overlap(token_match.span(), protected_span)
            for protected_span in protected_char_spans
        ):
            continue

        eligible_indices.append(i)

    return token_matches, cue_indices, eligible_indices

In [572]:
def mask_random_control_example(
    context_posts,
    cue_sequences,
    rng,
):
    """
    Mask the same number of eligible random word tokens as lexical-cue
    tokens within one complete user-target example.
    """

    post_infos = []
    eligible_positions = []
    requested_tokens = 0

    # First collect information across all context posts
    for post_idx, post in enumerate(context_posts):

        text = post.get("Content")

        token_matches, cue_indices, eligible_indices = get_control_info(
            text,
            cue_sequences,
        )

        requested_tokens += len(cue_indices)

        eligible_positions.extend(
            (post_idx, token_idx)
            for token_idx in eligible_indices
        )

        post_infos.append({
            "token_matches": token_matches,
        })

    # Examples unaffected by lexical masking remain unchanged
    if requested_tokens == 0:
        return (
            [post.copy() for post in context_posts],
            0,
            0,
            len(eligible_positions),
        )

    # Match the requested number whenever enough eligible tokens exist
    n_remove = min(
        requested_tokens,
        len(eligible_positions),
    )

    selected_positions = rng.choice(
        len(eligible_positions),
        size=n_remove,
        replace=False,
    )

    # Store selected token indices separately for each post
    selected_by_post = {}

    for selected_position in selected_positions:

        post_idx, token_idx = eligible_positions[
            int(selected_position)
        ]

        selected_by_post.setdefault(
            post_idx,
            []
        ).append(token_idx)

    masked_posts = []

    for post_idx, post in enumerate(context_posts):

        masked_post = post.copy()
        text = post.get("Content")

        selected_token_indices = selected_by_post.get(
            post_idx,
            [],
        )

        if isinstance(text, str) and selected_token_indices:

            token_matches = post_infos[
                post_idx
            ]["token_matches"]

            char_spans = [
                token_matches[token_idx].span()
                for token_idx in selected_token_indices
            ]

            masked_text = text

            # Replace right-to-left so character offsets remain valid
            for start, end in sorted(
                char_spans,
                reverse=True,
            ):
                masked_text = (
                    masked_text[:start]
                    + lexical_placeholder
                    + masked_text[end:]
                )

            masked_post["Content"] = masked_text

        masked_posts.append(masked_post)

    return (
        masked_posts,
        requested_tokens,
        n_remove,
        len(eligible_positions),
    )

In [573]:
def create_matched_random_control(
    df,
    cue_sequences_by_target,
    seed,
):
    """Create one matched random-control version of the test set."""

    rng = np.random.default_rng(seed)

    control_df = df.copy(deep=True)

    new_contexts = []
    audit_rows = []

    for example_idx, row in enumerate(
        df.itertuples(index=False)
    ):

        cue_sequences = cue_sequences_by_target[
            row.TargetEntity
        ]

        (
            masked_posts,
            requested_tokens,
            removed_tokens,
            available_control_tokens,
        ) = mask_random_control_example(
            row.ContextPosts,
            cue_sequences,
            rng,
        )

        new_contexts.append(masked_posts)

        audit_rows.append({
            "example": example_idx,
            "target": row.TargetEntity,
            "requested_tokens": requested_tokens,
            "available_control_tokens": available_control_tokens,
            "removed_tokens": removed_tokens,
            "shortfall_tokens": requested_tokens - removed_tokens,
            "exact_match": requested_tokens == removed_tokens,
        })

    control_df["ContextPosts"] = new_contexts

    audit_df = pd.DataFrame(audit_rows)

    return control_df, audit_df

In [574]:
control_seeds = [1, 2, 3, 4, 5]

top10_controls = {}
top10_control_audits = {}

top25_controls = {}
top25_control_audits = {}

for seed in control_seeds:

    (
        top10_controls[seed],
        top10_control_audits[seed],
    ) = create_matched_random_control(
        human_test,
        top10_cue_sequences_by_target,
        seed=seed,
    )

    (
        top25_controls[seed],
        top25_control_audits[seed],
    ) = create_matched_random_control(
        human_test,
        top25_cue_sequences_by_target,
        seed=seed,
    )

In [575]:
def summarize_control_audits(audits, variant):

    rows = []

    for seed, audit in audits.items():

        affected = audit[
            audit["requested_tokens"] > 0
        ]

        rows.append({
            "variant": variant,
            "seed": seed,
            "affected_examples": len(affected),
            "requested_tokens": affected["requested_tokens"].sum(),
            "removed_tokens": affected["removed_tokens"].sum(),
            "shortfall_tokens": affected["shortfall_tokens"].sum(),
            "exact_match_%": round(
                100 * affected["exact_match"].mean(),
                2,
            ) if len(affected) > 0 else 100.0,
        })

    return pd.DataFrame(rows)


control_summary = pd.concat(
    [
        summarize_control_audits(
            top10_control_audits,
            "Top 10",
        ),
        summarize_control_audits(
            top25_control_audits,
            "Top 25",
        ),
    ],
    ignore_index=True,
)

display(control_summary)

,variant,seed,affected_examples,requested_tokens,removed_tokens,shortfall_tokens,exact_match_%
0,Top 10,1,124,703,703,0,100.0
1,Top 10,2,124,703,703,0,100.0
2,Top 10,3,124,703,703,0,100.0
3,Top 10,4,124,703,703,0,100.0
4,Top 10,5,124,703,703,0,100.0
5,Top 25,1,265,1117,1117,0,100.0
6,Top 25,2,265,1117,1117,0,100.0
7,Top 25,3,265,1117,1117,0,100.0
8,Top 25,4,265,1117,1117,0,100.0
9,Top 25,5,265,1117,1117,0,100.0


In [576]:
for seed, control_df in top10_controls.items():
    control_df.to_parquet(
        "../data/interventions/"
        f"human_test_lexical_cues_top10_control_seed{seed}.parquet",
        index=False,
    )

for seed, control_df in top25_controls.items():
    control_df.to_parquet(
        "../data/interventions/"
        f"human_test_lexical_cues_top25_control_seed{seed}.parquet",
        index=False,
    )